# 01. Regresión Lineal

**Nivel:** 🟢 | **Tiempo:** 75-90 min | **Prereq:** Python, NumPy

## 📋 Tabla de Contenidos

1. [Motivación](#1-motivacion)
2. [Intuición Visual](#2-intuicion)
3. [Fundamentos Matemáticos](#3-fundamentos)
4. [Implementación Desde Cero](#4-implementacion)
5. [**🎯 Ejercicios GRADED**](#5-graded) ⭐
6. [Versión con Framework](#6-framework)
7. [Papers Fundamentales](#7-papers)
8. [Best Practices](#8-practices)
9. [Conclusiones](#9-conclusiones)
10. [Referencias y Navegación](#10-nav)

## 🎯 Objetivos de Aprendizaje

Al completar este notebook serás capaz de:
- Comprender los fundamentos matemáticos de regresión lineal
- Implementar regresión lineal desde cero usando NumPy
- Visualizar datos y resultados de forma profesional
- Aplicar gradient descent para optimización
- Evaluar modelos usando métricas apropiadas (MSE, R²)
- Usar scikit-learn para implementaciones en producción

---
## 1. 💡 Motivación

### ¿Por qué Regresión Lineal?

La regresión lineal es **el algoritmo fundamental de Machine Learning**. Aunque simple, es increíblemente poderoso y forma la base de técnicas más avanzadas como redes neuronales.

### 🌍 Casos de Uso Reales

**1. Predicción de Precios de Viviendas** 🏠
- **Input:** Tamaño (m²), ubicación, # habitaciones, edad
- **Output:** Precio estimado en USD/EUR
- **Usado por:** Zillow ($3B valuación), Idealista, Mercado Libre
- **Impacto:** Ayuda a millones a tomar decisiones de compra

**2. Forecasting de Ventas** 📈
- **Input:** Inversión marketing, temporada, precio, competencia
- **Output:** Ventas proyectadas próximo trimestre
- **Usado por:** Amazon, Walmart, toda empresa retail
- **Impacto:** Optimización de inventario ($Bn ahorrados)

**3. Análisis Financiero** 💰
- **Input:** Indicadores económicos, históricos
- **Output:** Rendimiento esperado de activos
- **Usado por:** Bloomberg, Goldman Sachs, fondos de inversión
- **Impacto:** Gestión de riesgo en portfolios de billones

**4. Medicina y Farmacéutica** 💊
- **Input:** Dosis medicamento, edad, peso, genética
- **Output:** Respuesta del paciente (niveles en sangre)
- **Usado por:** FDA, labs farmacéuticas
- **Impacto:** Dosificación personalizada, salva vidas

### 💼 Valor en la Industria

| Ventaja | Descripción | Ejemplo |
|---------|-------------|---------|
| **Interpretabilidad** | Coeficientes tienen significado claro | "Por cada m² adicional, precio sube $2,500" |
| **Velocidad** | Entrena en milisegundos incluso con 10M+ datos | Real-time pricing en e-commerce |
| **Baseline sólido** | Siempre empieza aquí antes de modelos complejos | Regla de oro en Kaggle |
| **Producción-ready** | Implementaciones ultra-optimizadas (C++/CUDA) | Latencia <1ms en servidores |

> **"En mi experiencia, regresión lineal bien aplicada supera a redes neuronales mal diseñadas el 70% del tiempo"**  
> — Andrew Ng, fundador de Coursera, ex-líder de Google Brain

### 🎯 Lo que aprenderás HOY

En este notebook no solo verás la teoría, sino que:
1. ✅ Implementarás gradient descent desde cero
2. ✅ Visualizarás el proceso de optimización paso a paso
3. ✅ Compararás tu implementación con scikit-learn
4. ✅ Aplicarás técnicas de regularización
5. ✅ Entenderás cuándo NO usar regresión lineal

In [ ]:
# Imports necesarios
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from typing import Tuple, List

# Configuración visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print('✅ Imports completados')
print(f'NumPy version: {np.__version__}')

---
## 2. 📊 Intuición Visual

### La Idea Central

Regresión lineal busca **la mejor línea recta** (o hiperplano en >2D) que pase por nuestros datos.

**Matemáticamente (caso simple):**
$$y = wx + b$$

**Caso general (múltiples features):**
$$y = w_1x_1 + w_2x_2 + ... + w_nx_n + b = \mathbf{w}^T\mathbf{x} + b$$

Donde:
- $y$ = predicción (variable dependiente)
- $\mathbf{x}$ = vector de inputs (variables independientes)
- $\mathbf{w}$ = vector de pesos (weights) - **parámetros a aprender**
- $b$ = intersección (bias) - **parámetro a aprender**

### 🎯 Objetivo

Encontrar $\mathbf{w}$ y $b$ que **minimicen el error cuadrático medio (MSE)**:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Donde $\hat{y}_i = \mathbf{w}^T\mathbf{x}_i + b$ es nuestra predicción.

In [ ]:
# Generar datos sintéticos para visualización
np.random.seed(42)
n_samples = 100

# Caso 1D: y = 2.5x + 1.5 + ruido
X_1d = np.linspace(0, 10, n_samples)
y_true_1d = 2.5 * X_1d + 1.5
y_noisy_1d = y_true_1d + np.random.randn(n_samples) * 3

print(f"📊 Generados {n_samples} puntos de datos")
print(f"📊 Relación verdadera: y = 2.5x + 1.5")
print(f"📊 Ruido añadido: σ = 3.0")

In [ ]:
# VISUALIZACIÓN 1: Proceso de regresión lineal paso a paso
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Subplot 1: Datos originales
ax = axes[0, 0]
ax.scatter(X_1d, y_noisy_1d, alpha=0.6, s=60, c='steelblue', edgecolors='navy')
ax.set_xlabel('X (Feature)', fontsize=12, fontweight='bold')
ax.set_ylabel('y (Target)', fontsize=12, fontweight='bold')
ax.set_title('Paso 1: Datos Observados (con ruido)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.95, 'n = 100 puntos', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
        verticalalignment='top', fontsize=10)

# Subplot 2: Con línea verdadera (oculta en la realidad)
ax = axes[0, 1]
ax.scatter(X_1d, y_noisy_1d, alpha=0.6, s=60, c='steelblue', label='Datos', edgecolors='navy')
ax.plot(X_1d, y_true_1d, 'g--', linewidth=3, label='Función real (desconocida)', alpha=0.7)
ax.set_xlabel('X (Feature)', fontsize=12, fontweight='bold')
ax.set_ylabel('y (Target)', fontsize=12, fontweight='bold')
ax.set_title('Paso 2: Relación Verdadera Oculta', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.95, 'y = 2.5x + 1.5', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
        verticalalignment='top', fontsize=10)

# Subplot 3: Predicción inicial (mala)
ax = axes[1, 0]
w_bad, b_bad = 1.0, 5.0  # Parámetros malos inicialmente
y_pred_bad = w_bad * X_1d + b_bad
ax.scatter(X_1d, y_noisy_1d, alpha=0.6, s=60, c='steelblue', label='Datos', edgecolors='navy')
ax.plot(X_1d, y_pred_bad, 'r-', linewidth=3, label=f'Predicción mala: y={w_bad}x+{b_bad}')
mse_bad = np.mean((y_noisy_1d - y_pred_bad)**2)
ax.set_xlabel('X (Feature)', fontsize=12, fontweight='bold')
ax.set_ylabel('y (Target)', fontsize=12, fontweight='bold')
ax.set_title('Paso 3: Primer Intento (w=1.0, b=5.0)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.95, f'MSE = {mse_bad:.2f}\n(muy alto!)', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='salmon', alpha=0.8),
        verticalalignment='top', fontsize=10)

# Subplot 4: Predicción optimizada
ax = axes[1, 1]
# Calcular óptimos usando la fórmula cerrada
X_with_bias = np.c_[X_1d, np.ones(n_samples)]
w_opt = np.linalg.lstsq(X_with_bias, y_noisy_1d, rcond=None)[0]
w_final, b_final = w_opt[0], w_opt[1]
y_pred_opt = w_final * X_1d + b_final

ax.scatter(X_1d, y_noisy_1d, alpha=0.6, s=60, c='steelblue', label='Datos', edgecolors='navy')
ax.plot(X_1d, y_true_1d, 'g--', linewidth=2, label='Real', alpha=0.4)
ax.plot(X_1d, y_pred_opt, 'purple', linewidth=3, 
        label=f'Óptimo: y={w_final:.2f}x+{b_final:.2f}')
mse_opt = np.mean((y_noisy_1d - y_pred_opt)**2)
ax.set_xlabel('X (Feature)', fontsize=12, fontweight='bold')
ax.set_ylabel('y (Target)', fontsize=12, fontweight='bold')
ax.set_title('Paso 4: Después de Optimización', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.95, f'MSE = {mse_opt:.2f}\n(minimizado!)', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
        verticalalignment='top', fontsize=10)

plt.tight_layout()
plt.show()

print(f'\n📊 COMPARACIÓN DE MODELOS:')
print(f'{"="*50}')
print(f'Parámetros verdaderos:  w=2.50, b=1.50')
print(f'Predicción mala:        w={w_bad:.2f}, b={b_bad:.2f} → MSE={mse_bad:.2f}')
print(f'Predicción optimizada:  w={w_final:.2f}, b={b_final:.2f} → MSE={mse_opt:.2f}')
print(f'\n💡 ¡Reducción de error: {((mse_bad - mse_opt)/mse_bad * 100):.1f}%!')

In [ ]:
# VISUALIZACIÓN 2: Residuales (errores)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Residuals visualization
ax = axes[0]
ax.scatter(X_1d, y_noisy_1d, alpha=0.6, s=80, c='blue', label='Datos', zorder=3)
ax.plot(X_1d, y_pred_opt, 'r-', linewidth=3, label=f'Modelo óptimo', zorder=2)

# Draw error lines
for i in range(0, len(X_1d), 5):  # Every 5th point for clarity
    ax.plot([X_1d[i], X_1d[i]], [y_noisy_1d[i], y_pred_opt[i]], 
            'k--', alpha=0.3, linewidth=1.5, zorder=1)

ax.set_xlabel('X', fontsize=13, fontweight='bold')
ax.set_ylabel('y', fontsize=13, fontweight='bold')
ax.set_title('Residuales: Diferencia entre Dato y Predicción', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

residuals = y_noisy_1d - y_pred_opt
ax.text(0.02, 0.98, f'MSE = {np.mean(residuals**2):.2f}\nRMSE = {np.sqrt(np.mean(residuals**2)):.2f}', 
        transform=ax.transAxes, fontsize=12, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

# Right: Residual distribution
ax = axes[1]
ax.hist(residuals, bins=20, alpha=0.7, color='skyblue', edgecolor='navy')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Error = 0')
ax.set_xlabel('Residual (y_real - y_pred)', fontsize=13, fontweight='bold')
ax.set_ylabel('Frecuencia', fontsize=13, fontweight='bold')
ax.set_title('Distribución de Residuales', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Stats
ax.text(0.02, 0.98, f'Media: {np.mean(residuals):.3f}\nStd: {np.std(residuals):.3f}', 
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9))

plt.tight_layout()
plt.show()

print('\n🔍 ANÁLISIS DE RESIDUALES:')
print(f'{"="*60}')
print(f'Media de residuales: {np.mean(residuals):.4f} (debería ser ≈0)')
print(f'Std de residuales:   {np.std(residuals):.4f}')
print(f'Min residual:        {np.min(residuals):.4f}')
print(f'Max residual:        {np.max(residuals):.4f}')
print(f'\n💡 Si residuales ~Normal(0, σ²), el modelo es adecuado')

---
## 3. 📐 Fundamentos Matemáticos

### 3.1 Formulación del Problema

**Dado:**
- Dataset: $\{(\mathbf{x}_1, y_1), (\mathbf{x}_2, y_2), ..., (\mathbf{x}_n, y_n)\}$
- Donde $\mathbf{x}_i \in \mathbb{R}^d$ (features) y $y_i \in \mathbb{R}$ (target)

**Modelo:**
$$\hat{y}_i = \mathbf{w}^T\mathbf{x}_i + b = \sum_{j=1}^{d} w_j x_{ij} + b$$

**Objetivo:** Encontrar $\mathbf{w}^* $ y $b^*$ que minimicen:

$$J(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2 = \frac{1}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i - b)^2$$

### 3.2 Solución Analítica (Normal Equations)

Si agregamos $b$ como una feature adicional (trick del bias):
$$\mathbf{X} \in \mathbb{R}^{n \times (d+1)}, \quad \mathbf{w} \in \mathbb{R}^{d+1}$$

**Solución en forma cerrada:**
$$\mathbf{w}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

**Derivación:**
1. $J(\mathbf{w}) = ||\mathbf{y} - \mathbf{X}\mathbf{w}||^2$
2. $\frac{\partial J}{\partial \mathbf{w}} = -2\mathbf{X}^T(\mathbf{y} - \mathbf{X}\mathbf{w})$
3. Igualar a 0: $\mathbf{X}^T\mathbf{y} = \mathbf{X}^T\mathbf{X}\mathbf{w}$
4. Despejar: $\mathbf{w}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$

**Limitaciones:**
- ⚠️ Requiere invertir $\mathbf{X}^T\mathbf{X}$ → $O(d^3)$ complejidad
- ⚠️ Problemas si $\mathbf{X}^T\mathbf{X}$ no es invertible (multicolinealidad)
- ⚠️ No escalable a datasets masivos (>1M muestras)

### 3.3 Solución Iterativa (Gradient Descent)

**Idea:** Moverse en dirección opuesta al gradiente

**Algoritmo:**
1. Inicializar $\mathbf{w}^{(0)}$ aleatoriamente
2. Para $t = 1, 2, ..., T$:
   $$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \alpha \nabla J(\mathbf{w}^{(t)})$$

**Gradiente:**
$$\nabla J(\mathbf{w}) = \frac{2}{n}\mathbf{X}^T(\mathbf{X}\mathbf{w} - \mathbf{y})$$

**Componentes:**
$$\frac{\partial J}{\partial w_j} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)x_{ij}$$

$$\frac{\partial J}{\partial b} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)$$

Donde $\alpha$ es el **learning rate** (hiperparámetro crítico).

### 3.4 Métricas de Evaluación

**1. Mean Squared Error (MSE)**
$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

**2. Root Mean Squared Error (RMSE)**
$$\text{RMSE} = \sqrt{\text{MSE}}$$
- ✅ Mismas unidades que $y$

**3. Mean Absolute Error (MAE)**
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$
- ✅ Más robusto a outliers que MSE

**4. R² Score (Coeficiente de Determinación)**
$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

- $R^2 = 1$: Predicción perfecta
- $R^2 = 0$: Modelo tan bueno como predecir la media
- $R^2 < 0$: Modelo peor que la media (¡malo!)

**5. Adjusted R²**
$$R^2_{adj} = 1 - \frac{(1-R^2)(n-1)}{n-d-1}$$
- ✅ Penaliza features innecesarias

---
## 4. 💻 Implementación Desde Cero

Vamos a implementar regresión lineal usando **solo NumPy**, sin scikit-learn ni bibliotecas de ML.

### 4.1 Clase LinearRegression

Implementaremos dos métodos:
1. **Normal Equations** (solución analítica)
2. **Gradient Descent** (solución iterativa)

In [ ]:
class LinearRegressionFromScratch:
    """
    Regresión Lineal implementada desde cero con NumPy.
    
    Soporta dos métodos de optimización:
    - 'normal': Ecuaciones normales (solución cerrada)
    - 'gd': Gradient Descent (iterativo)
    """
    
    def __init__(self, method='gd', learning_rate=0.01, n_iterations=1000):
        """
        Parameters:
        -----------
        method : str, default='gd'
            Método de optimización: 'normal' o 'gd'
        learning_rate : float, default=0.01
            Learning rate para gradient descent
        n_iterations : int, default=1000
            Número de iteraciones para gradient descent
        """
        self.method = method
        self.lr = learning_rate
        self.n_iters = n_iterations
        self.w = None  # Pesos
        self.b = None  # Bias
        self.losses = []  # Historial de pérdidas
        
    def fit(self, X, y):
        """
        Entrenar el modelo.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Training data
        y : np.ndarray, shape (n_samples,)
            Target values
        """
        n_samples, n_features = X.shape
        
        if self.method == 'normal':
            # Método 1: Ecuaciones Normales
            X_b = np.c_[X, np.ones(n_samples)]  # Agregar columna de 1s para bias
            # w_opt = (X^T X)^-1 X^T y
            theta = np.linalg.lstsq(X_b.T @ X_b, X_b.T @ y, rcond=None)[0]
            self.w = theta[:-1]
            self.b = theta[-1]
            
            # Calcular loss final
            y_pred = self.predict(X)
            final_loss = np.mean((y - y_pred) ** 2)
            self.losses = [final_loss]
            
        elif self.method == 'gd':
            # Método 2: Gradient Descent
            # Inicializar parámetros
            self.w = np.zeros(n_features)
            self.b = 0
            
            # Gradient Descent
            for i in range(self.n_iters):
                # Predicción actual
                y_pred = X @ self.w + self.b
                
                # Calcular gradientes
                dw = (2/n_samples) * (X.T @ (y_pred - y))
                db = (2/n_samples) * np.sum(y_pred - y)
                
                # Actualizar parámetros
                self.w -= self.lr * dw
                self.b -= self.lr * db
                
                # Guardar pérdida
                loss = np.mean((y - y_pred) ** 2)
                self.losses.append(loss)
                
                # Logging cada 100 iteraciones
                if (i + 1) % 100 == 0:
                    print(f'Iteración {i+1}/{self.n_iters}, Loss: {loss:.4f}')
        
        return self
    
    def predict(self, X):
        """
        Hacer predicciones.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Test data
            
        Returns:
        --------
        y_pred : np.ndarray, shape (n_samples,)
            Predicted values
        """
        return X @ self.w + self.b
    
    def score(self, X, y):
        """
        Calcular R² score.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Test data
        y : np.ndarray, shape (n_samples,)
            True values
            
        Returns:
        --------
        r2 : float
            R² score
        """
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - (ss_res / ss_tot)
        return r2

print('✅ Clase LinearRegressionFromScratch implementada')
print('   Métodos disponibles: normal, gd')

In [ ]:
# DEMO 1: Entrenar con Gradient Descent
print('='*70)
print('DEMO 1: GRADIENT DESCENT')
print('='*70)

# Generar datos de prueba
np.random.seed(42)
X_train = np.random.randn(200, 1) * 2
y_train = 3 * X_train.squeeze() + 7 + np.random.randn(200) * 1.5

# Entrenar modelo
model_gd = LinearRegressionFromScratch(method='gd', learning_rate=0.1, n_iterations=500)
model_gd.fit(X_train, y_train)

# Resultados
print(f'\n📊 RESULTADOS FINALES:')
print(f'Peso estimado (w): {model_gd.w[0]:.4f} (real: 3.0000)')
print(f'Bias estimado (b): {model_gd.b:.4f} (real: 7.0000)')
print(f'Loss final:        {model_gd.losses[-1]:.4f}')
print(f'R² score:          {model_gd.score(X_train, y_train):.4f}')

In [ ]:
# DEMO 2: Comparar Normal Equations vs Gradient Descent
print('='*70)
print('DEMO 2: COMPARACIÓN DE MÉTODOS')
print('='*70)

# Entrenar con Normal Equations
import time

start = time.time()
model_normal = LinearRegressionFromScratch(method='normal')
model_normal.fit(X_train, y_train)
time_normal = time.time() - start

# Entrenar con Gradient Descent
start = time.time()
model_gd2 = LinearRegressionFromScratch(method='gd', learning_rate=0.1, n_iterations=1000)
model_gd2.fit(X_train, y_train)
time_gd = time.time() - start

print(f'\n📊 COMPARACIÓN:')
print(f'{"":<20} {"Normal Eq":<15} {"Gradient Descent":<15}')
print(f'{"-"*50}')
print(f'{"Peso (w):":<20} {model_normal.w[0]:<15.4f} {model_gd2.w[0]:<15.4f}')
print(f'{"Bias (b):":<20} {model_normal.b:<15.4f} {model_gd2.b:<15.4f}')
print(f'{"R² score:":<20} {model_normal.score(X_train, y_train):<15.4f} {model_gd2.score(X_train, y_train):<15.4f}')
print(f'{"Tiempo (s):":<20} {time_normal:<15.6f} {time_gd:<15.6f}')

print(f'\n💡 Normal Equations es más rápido para datasets pequeños')
print(f'💡 Gradient Descent escala mejor para datasets grandes (>10K muestras)')

In [ ]:
# VISUALIZACIÓN: Convergencia de Gradient Descent
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Loss curve
ax = axes[0]
ax.plot(model_gd2.losses, linewidth=2, color='darkblue')
ax.set_xlabel('Iteración', fontsize=13, fontweight='bold')
ax.set_ylabel('MSE Loss', fontsize=13, fontweight='bold')
ax.set_title('Convergencia de Gradient Descent', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')  # Log scale para ver mejor

# Anotar puntos clave
ax.annotate(f'Inicio: {model_gd2.losses[0]:.2f}', 
            xy=(0, model_gd2.losses[0]), xytext=(100, model_gd2.losses[0]*1.5),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=11, color='red')
ax.annotate(f'Final: {model_gd2.losses[-1]:.2f}', 
            xy=(len(model_gd2.losses)-1, model_gd2.losses[-1]), 
            xytext=(len(model_gd2.losses)-200, model_gd2.losses[-1]*2),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=11, color='green')

# Right: Predictions vs True
ax = axes[1]
y_pred = model_gd2.predict(X_train)
ax.scatter(y_train, y_pred, alpha=0.5, s=40, c='steelblue', edgecolors='navy')
ax.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
        'r--', linewidth=3, label='Predicción perfecta')
ax.set_xlabel('y real', fontsize=13, fontweight='bold')
ax.set_ylabel('y predicho', fontsize=13, fontweight='bold')
ax.set_title('Predicciones vs Valores Reales', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

r2 = model_gd2.score(X_train, y_train)
ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes,
        fontsize=13, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9))

plt.tight_layout()
plt.show()

print(f'✅ Modelo converge en ~{len(model_gd2.losses)} iteraciones')

---
## 5. 🎯 Ejercicios GRADED

**Instrucciones Generales:**
1. Completa el código entre `# START CODE HERE` y `# END CODE HERE`
2. Ejecuta la celda de test para verificar tu implementación
3. Usa los hints si necesitas ayuda
4. **No modifiques** la firma de las funciones

**Sistema de Puntos:**
- Total: 100 puntos
- Mínimo para aprobar: 70 puntos
- Tests automáticos en: `tests/test_01.py`

### Ejercicio 1: Implementar MSE (15 pts)

Implementa la función `implement_mse` que calcula el **Mean Squared Error**.

**Fórmula:**
$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_{true_i} - y_{pred_i})^2$$

**Casos de uso:**
- Métrica principal para regresión lineal
- Penaliza errores grandes (por el cuadrado)
- Siempre positivo

In [ ]:
def implement_mse(y_true, y_pred):
    """
    Calcular Mean Squared Error.
    
    Parameters:
    -----------
    y_true : np.ndarray, shape (n_samples,)
        Valores reales
    y_pred : np.ndarray, shape (n_samples,)
        Valores predichos
        
    Returns:
    --------
    mse : float
        Mean squared error
    """
    # START CODE HERE (≈ 2-3 líneas)
    # 1. Calcular diferencias (errores)
    # 2. Elevar al cuadrado
    # 3. Calcular la media
    mse = None  # Tu código aquí
    # END CODE HERE
    return mse

<details><summary>💡 Hint 1: Fórmula paso a paso</summary>

1. Calcula las diferencias: `errors = y_true - y_pred`
2. Eleva al cuadrado: `squared_errors = errors ** 2`
3. Calcula la media: `mse = np.mean(squared_errors)`

O en una línea: `mse = np.mean((y_true - y_pred) ** 2)`
</details>

<details><summary>💡 Hint 2: Funciones de NumPy útiles</summary>

- `np.mean()`: Calcula la media de un array
- `**2` o `np.square()`: Eleva al cuadrado
- `np.subtract()` o `-`: Resta elemento a elemento
</details>

<details><summary>🔑 Solución Completa</summary>

```python
def implement_mse(y_true, y_pred):
    """Calcular Mean Squared Error"""
    # Método 1: Paso a paso (más claro)
    errors = y_true - y_pred
    squared_errors = errors ** 2
    mse = np.mean(squared_errors)
    return mse
    
    # Método 2: Una línea (más conciso)
    # return np.mean((y_true - y_pred) ** 2)
```

**Explicación:**
- `y_true - y_pred`: Vector de errores (residuales)
- `** 2`: Cuadrado elemento a elemento
- `np.mean()`: Promedio de todos los errores cuadráticos
</details>

In [ ]:
# 🧪 Test de implement_mse
print('Testing implement_mse...')
print('-' * 60)

# Test 1: Caso simple
y_true_test = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred_test = np.array([1.1, 2.1, 2.9, 4.2, 4.8])
result = implement_mse(y_true_test, y_pred_test)
expected = 0.026
print(f'Test 1: MSE de predicciones con error pequeño')
print(f'  y_true: {y_true_test}')
print(f'  y_pred: {y_pred_test}')
print(f'  Result: {result:.6f}')
print(f'  Expected: ~{expected:.6f}')
print(f'  ✅ PASS' if abs(result - expected) < 0.01 else f'  ❌ FAIL')

# Test 2: Predicción perfecta
y_true_test2 = np.array([1, 2, 3])
y_pred_test2 = np.array([1, 2, 3])
result2 = implement_mse(y_true_test2, y_pred_test2)
print(f'\nTest 2: Predicción perfecta (MSE debe ser 0)')
print(f'  Result: {result2:.6f}')
print(f'  ✅ PASS' if result2 == 0 else f'  ❌ FAIL')

# Test 3: Error grande
y_true_test3 = np.array([10, 20, 30])
y_pred_test3 = np.array([15, 25, 35])
result3 = implement_mse(y_true_test3, y_pred_test3)
expected3 = 25.0
print(f'\nTest 3: Error constante de 5 unidades')
print(f'  Result: {result3:.6f}')
print(f'  Expected: {expected3:.6f}')
print(f'  ✅ PASS' if abs(result3 - expected3) < 0.01 else f'  ❌ FAIL')

print(f'\n✅ Tests completados! Puntos: 15/15' if all([
    abs(result - expected) < 0.01,
    result2 == 0,
    abs(result3 - expected3) < 0.01
]) else '❌ Algunos tests fallaron')

### Ejercicio 2: Calcular Gradiente (20 pts)

Implementa `compute_gradient` que calcula el gradiente de la función de pérdida MSE.

**Gradientes:**
- $\frac{\partial J}{\partial w} = \frac{2}{n}X^T(X w - y)$
- $\frac{\partial J}{\partial b} = \frac{2}{n}\sum(\hat{y} - y)$

In [ ]:
def compute_gradient(X, y, w, b):
    """
    Calcular gradientes para regresión lineal.
    
    Parameters:
    -----------
    X : np.ndarray, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    w : np.ndarray, shape (n_features,)
    b : float
        
    Returns:
    --------
    dw : np.ndarray, shape (n_features,) - gradiente respecto a w
    db : float - gradiente respecto a b
    """
    # START CODE HERE (≈ 4-6 líneas)
    n = len(y)
    # 1. Calcular predicciones: y_pred = X @ w + b
    # 2. Calcular error: error = y_pred - y  
    # 3. dw = (2/n) * X^T @ error
    # 4. db = (2/n) * sum(error)
    dw = None
    db = None
    # END CODE HERE
    return dw, db

<details><summary>💡 Hint</summary>

```python
n = len(y)
y_pred = X @ w + b
error = y_pred - y
dw = (2/n) * (X.T @ error)
db = (2/n) * np.sum(error)
```
</details>

<details><summary>🔑 Solución</summary>

```python
def compute_gradient(X, y, w, b):
    n = len(y)
    y_pred = X @ w + b
    error = y_pred - y
    dw = (2/n) * (X.T @ error)
    db = (2/n) * np.sum(error)
    return dw, db
```
</details>

In [ ]:
# 🧪 Test
print('Testing compute_gradient...')
X_test = np.array([[1], [2], [3]])
y_test = np.array([2, 4, 6])
w_test, b_test = np.array([1.5]), 0.5
dw, db = compute_gradient(X_test, y_test, w_test, b_test)
print(f'dw: {dw}, db: {db}')
print('Expected: dw ≈ [0.0], db ≈ 0.0 (ya que y=2x)')
print('✅ +20 pts' if np.allclose(dw, [0.0], atol=1) and np.isclose(db, 0.0, atol=1) else '❌')

### Ejercicio 3: Entrenar Modelo (30 pts)

Implementa `fit_linear_model` usando gradient descent completo.

In [ ]:
def fit_linear_model(X_train, y_train, learning_rate=0.01, epochs=1000):
    """
    Entrenar regresión lineal con gradient descent.
    
    Returns:
    --------
    w, b : parámetros entrenados
    losses : lista de pérdidas por época
    """
    # START CODE HERE (≈ 10-15 líneas)
    n_samples, n_features = X_train.shape
    w = np.zeros(n_features)
    b = 0
    losses = []
    
    for epoch in range(epochs):
        # 1. Calcular predicciones
        # 2. Calcular gradientes usando compute_gradient
        # 3. Actualizar w y b
        # 4. Guardar loss
        pass
    
    # END CODE HERE
    return w, b, losses

<details><summary>🔑 Solución</summary>

```python
def fit_linear_model(X_train, y_train, learning_rate=0.01, epochs=1000):
    n_samples, n_features = X_train.shape
    w = np.zeros(n_features)
    b = 0
    losses = []
    
    for epoch in range(epochs):
        dw, db = compute_gradient(X_train, y_train, w, b)
        w -= learning_rate * dw
        b -= learning_rate * db
        
        y_pred = X_train @ w + b
        loss = implement_mse(y_train, y_pred)
        losses.append(loss)
    
    return w, b, losses
```
</details>

In [ ]:
# 🧪 Test
X_test = np.random.randn(100, 2)
y_test = 3*X_test[:, 0] + 2*X_test[:, 1] + 1 + np.random.randn(100)*0.1
w, b, losses = fit_linear_model(X_test, y_test, learning_rate=0.1, epochs=500)
print(f'Trained w: {w}, b: {b:.2f}')
print(f'Expected: w≈[3, 2], b≈1')
print('✅ +30 pts' if losses[-1] < 1.0 else '❌')

### Ejercicio 4: Predicción (15 pts)

In [ ]:
def predict(X, w, b):
    """Hacer predicciones: y = Xw + b"""
    # START CODE HERE (≈ 1 línea)
    y_pred = None
    # END CODE HERE
    return y_pred

<details><summary>🔑 Solución</summary>

```python
return X @ w + b
```
</details>

In [ ]:
# Test
print('Testing predict...')
y_pred = predict(X_test, w, b)
print(f'Shape: {y_pred.shape}, Expected: {y_test.shape}')
print('✅ +15 pts' if y_pred.shape == y_test.shape else '❌')

### Ejercicio 5: R² Score (20 pts)

In [ ]:
def compute_r2(y_true, y_pred):
    """Calcular R²: 1 - SS_res/SS_tot"""
    # START CODE HERE (≈ 3-4 líneas)
    # SS_res = sum((y_true - y_pred)²)
    # SS_tot = sum((y_true - mean(y_true))²)
    r2 = None
    # END CODE HERE
    return r2

<details><summary>🔑 Solución</summary>

```python
ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2 = 1 - (ss_res / ss_tot)
return r2
```
</details>

In [ ]:
# Test
r2 = compute_r2(y_test, y_pred)
print(f'R² score: {r2:.4f}')
print('✅ +20 pts' if r2 > 0.9 else '❌')

---
## 6. 🔧 Versión con Framework (scikit-learn)

En producción, usamos bibliotecas optimizadas como **scikit-learn**.

### Ventajas de scikit-learn:
- ✅ Implementación en C/Cython (100x más rápido)
- ✅ Manejo automático de edge cases
- ✅ API consistente con otros modelos
- ✅ Herramientas adicionales (cross-validation, pipelines, etc.)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Generar datos
np.random.seed(42)
X_demo = np.random.randn(1000, 5)
y_demo = X_demo @ np.array([3, -2, 1, 4, -1]) + 5 + np.random.randn(1000) * 0.5

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_demo, y_demo, test_size=0.2, random_state=42)

# Entrenar
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

# Evaluar
y_pred_sklearn = model_sklearn.predict(X_test)
mse_sklearn = mean_squared_error(y_test, y_pred_sklearn)
r2_sklearn = r2_score(y_test, y_pred_sklearn)

print('='*70)
print('SCIKIT-LEARN LINEAR REGRESSION')
print('='*70)
print(f'Coeficientes: {model_sklearn.coef_}')
print(f'Intercept: {model_sklearn.intercept_:.4f}')
print(f'MSE (test): {mse_sklearn:.6f}')
print(f'R² (test): {r2_sklearn:.6f}')
print(f'\n✅ Modelo entrenado en {X_train.shape[0]} muestras')
print(f'✅ Evaluado en {X_test.shape[0]} muestras')

In [ ]:
# Comparación visual: sklearn vs nuestra implementación
fig, ax = plt.subplots(figsize=(10, 6))

# Nuestra implementación
X_simple = X_test[:, 0].reshape(-1, 1)  # Solo primera feature para visualizar
model_ours = LinearRegressionFromScratch(method='gd', learning_rate=0.1, n_iterations=1000)
model_ours.fit(X_simple, y_test)
y_pred_ours = model_ours.predict(X_simple)

# sklearn
model_sk = LinearRegression()
model_sk.fit(X_simple, y_test)
y_pred_sk = model_sk.predict(X_simple)

# Plot
idx_sorted = np.argsort(X_simple.squeeze())
ax.scatter(X_simple, y_test, alpha=0.4, s=30, label='Datos reales')
ax.plot(X_simple[idx_sorted], y_pred_ours[idx_sorted], 'r-', linewidth=2, label='Nuestra Impl.')
ax.plot(X_simple[idx_sorted], y_pred_sk[idx_sorted], 'g--', linewidth=2, label='scikit-learn')
ax.set_xlabel('X', fontsize=12, fontweight='bold')
ax.set_ylabel('y', fontsize=12, fontweight='bold')
ax.set_title('Comparación: Nuestra Implementación vs scikit-learn', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

print(f'\n📊 Nuestra implementación: MSE={implement_mse(y_test, y_pred_ours):.6f}')
print(f'📊 scikit-learn:           MSE={mean_squared_error(y_test, y_pred_sk):.6f}')
print(f'\n💡 Resultados casi idénticos!')

plt.tight_layout()
plt.show()

---
## 7. 📄 Papers Fundamentales

### Paper 1: Least Squares and Linear Regression
**Legendre (1805) / Gauss (1809)**

**Contribución Histórica:**
- ✨ Primera formalización del método de mínimos cuadrados
- ✨ Base de toda la estadística moderna
- ✨ Usado originalmente para astronomía (órbitas planetarias)

**Impacto:** >200 años siendo uno de los métodos más usados en ciencia.

---

### Paper 2: Ridge Regression (L2 Regularization)
**Hoerl & Kennard (1970)**  
*Ridge Regression: Biased Estimation for Nonorthogonal Problems*

**Contribución:**
- Solución al problema de multicolinealidad
- Introduce penalización L2: $J(w) = MSE + \lambda ||w||^2$
- Trade-off sesgo-varianza

**Cuándo usar:**
- Features correlacionadas
- Prevenir overfitting
- Datasets con más features que muestras

---

### Paper 3: LASSO Regression (L1 Regularization)
**Tibshirani (1996)**  
[Journal of the Royal Statistical Society](https://www.jstor.org/stable/2346178)

**Contribución:**
- Penalización L1: $J(w) = MSE + \lambda ||w||_1$
- **Feature selection automática** (algunos w → 0)
- Interpretabilidad mejorada

**Aplicaciones:**
- Genómica (miles de genes, pocas muestras)
- Text mining
- Modelos sparse

---

### Paper 4: Gradient Descent for Machine Learning
**Rumelhart, Hinton & Williams (1986)**  
*Learning representations by back-propagating errors*

**Contribución:**
- Popularizó gradient descent para ML
- Base del entrenamiento de redes neuronales
- Algoritmo de backpropagation

**Impacto:** Fundamento del Deep Learning moderno.

---

### Paper 5: Practical Recommendations
**Goodfellow, Bengio & Courville (2016)**  
*Deep Learning* - Capítulo 5

**Temas cubiertos:**
- ✅ Cómo elegir learning rate
- ✅ Normalización de features
- ✅ Diagnóstico de underfitting/overfitting
- ✅ Regularización en práctica

**Link:** [www.deeplearningbook.org](https://www.deeplearningbook.org/)

---
## 8. ✅ Best Practices

### 8.1 Preprocesamiento de Datos

#### ✅ DO:

**1. Normalizar Features**
```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```
**Por qué:** Evita que features con rangos grandes dominen el gradient descent.

**2. Verificar Valores Faltantes**
```python
assert not np.isnan(X).any(), "Hay NaNs en X"
assert not np.isnan(y).any(), "Hay NaNs en y"
```

**3. Detectar Outliers**
```python
from scipy import stats
z_scores = np.abs(stats.zscore(X))
X_clean = X[(z_scores < 3).all(axis=1)]
```

#### ❌ DON'T:

- ❌ **NO normalizar después de split** → Data leakage
- ❌ **NO ignorar outliers** → Pueden destruir el modelo
- ❌ **NO usar features categóricas sin encodear** → Usar One-Hot o Label Encoding

---

### 8.2 Entrenamiento

#### ✅ DO:

**1. Usar Train/Test Split**
```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

**2. Validar Convergencia**
```python
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Loss')
# Debe bajar monotónicamente
```

**3. Tune Learning Rate**
```python
# Muy alto → oscilaciones
# Muy bajo → converge muy lento
# Probar: 0.001, 0.01, 0.1, 1.0
```

**4. Early Stopping**
```python
if abs(losses[-1] - losses[-2]) < 1e-6:
    print(f'Converged at iteration {i}')
    break
```

#### ❌ DON'T:

- ❌ **NO entrenar en todo el dataset** → Usar train/val/test split
- ❌ **NO usar learning rate fijo siempre** → Considerar learning rate decay
- ❌ **NO ignorar la escala temporal** → GD puede tardar mucho en datasets grandes

---

### 8.3 Evaluación

#### ✅ DO:

**1. Usar Múltiples Métricas**
```python
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)  # Más robusto a outliers
r2 = r2_score(y_test, y_pred)  # Interpretabilidad
```

**2. Validación Cruzada**
```python
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f'R² medio: {scores.mean():.3f} (+/- {scores.std():.3f})')
```

**3. Analizar Residuales**
```python
residuals = y_test - y_pred
plt.scatter(y_pred, residuals)
# Deben estar centrados en 0, sin patrón
```

#### ❌ DON'T:

- ❌ **NO usar solo MSE** → Complementar con R² y MAE
- ❌ **NO evaluar en datos de entrenamiento** → Overfitting falso
- ❌ **NO ignorar distribución de residuales** → Indica problemas del modelo

---

### 8.4 Cuándo NO usar Regresión Lineal

#### ❌ Relaciones No Lineales
```python
# Mal ejemplo: y = x²
X = np.linspace(-5, 5, 100)
y = X**2
# Regresión lineal fallará miserablemente
```
**Solución:** Polynomial regression, splines, o modelos no lineales (Random Forest, XGBoost)

#### ❌ Outliers Severos
```python
# Algunos puntos con errores enormes destruyen el modelo
```
**Solución:** 
- RANSAC regression (robusto a outliers)
- Huber loss (en vez de MSE)
- Limpiar outliers primero

#### ❌ Features Altamente Correlacionadas (Multicolinealidad)
```python
# Ejemplo: X1 = altura en cm, X2 = altura en pulgadas
# Coeficientes inestables, difíciles de interpretar
```
**Solución:**
- Ridge regression (L2)
- Eliminar features redundantes
- PCA para decorrelacionar

#### ❌ Muchas Features, Pocas Muestras (p >> n)
```python
# Ejemplo: 10,000 genes, 50 pacientes
# Modelo sobreajustará
```
**Solución:**
- LASSO regression (L1) para feature selection
- Ridge regression
- Métodos de reducción de dimensionalidad

---

### 8.5 Checklist de Producción

Antes de deployar tu modelo:

- [ ] ✅ Normalización de features implementada y guardada
- [ ] ✅ Validación cruzada realizada (R² > umbral aceptable)
- [ ] ✅ Residuales verificados (sin patrones claros)
- [ ] ✅ Outliers manejados
- [ ] ✅ Modelo serializado (pickle o joblib)
- [ ] ✅ Tests unitarios para predict()
- [ ] ✅ Monitoreo de drift en producción
- [ ] ✅ Fallback si input está fuera de rango de entrenamiento

---
## 9. 🎓 Conclusiones

### Conceptos Clave Aprendidos

#### 1. Matemática
- ✅ Regresión lineal minimiza error cuadrático medio (MSE)
- ✅ Solución analítica: $(X^TX)^{-1}X^Ty$ (Normal Equations)
- ✅ Solución iterativa: Gradient Descent
- ✅ Gradiente: $\nabla J = \frac{2}{n}X^T(Xw - y)$

#### 2. Implementación
- ✅ Implementamos desde cero usando solo NumPy
- ✅ Visualizamos convergencia de gradient descent
- ✅ Comparamos con scikit-learn (resultados idénticos)

#### 3. Evaluación
- ✅ Métricas: MSE, RMSE, MAE, R²
- ✅ Análisis de residuales para validar supuestos
- ✅ Train/test split para evitar overfitting

#### 4. Práctica
- ✅ Preprocesamiento (normalización, outliers)
- ✅ Hiperparámetros (learning rate, iteraciones)
- ✅ Cuándo NO usar regresión lineal

---

### Lo que hace a Regresión Lineal Especial

| Ventaja | Descripción |
|---------|-------------|
| 🎯 **Interpretabilidad** | Coeficientes indican impacto de cada feature |
| ⚡ **Velocidad** | Entrena en milisegundos, predice en microsegundos |
| 📊 **Baseline perfecto** | Siempre empieza aquí antes de modelos complejos |
| 🔬 **Teoría sólida** | Matemática bien entendida desde hace 200+ años |
| 🏭 **Producción-ready** | Implementaciones ultra-optimizadas disponibles |

---

### Próximos Pasos en tu Aprendizaje

**Notebook 02: Gradient Descent en Profundidad**
- Variantes: Batch, Stochastic, Mini-batch
- Momentum, AdaGrad, Adam
- Learning rate scheduling

**Extensiones de Regresión Lineal:**
1. **Polynomial Regression** - Relaciones no lineales
2. **Ridge & LASSO** - Regularización L1/L2
3. **Elastic Net** - Combinación de Ridge y LASSO
4. **Logistic Regression** - Clasificación binaria

**Modelos Alternativos:**
- Decision Trees → No lineales, sin necesidad de normalización
- Random Forest → Ensemble de árboles, muy robusto
- Gradient Boosting (XGBoost) → State-of-the-art para datos tabulares
- Neural Networks → Cuando tienes millones de datos y relaciones muy complejas

---

### Recursos Adicionales

**Libros:**
- 📘 *The Elements of Statistical Learning* (Hastie et al.) - Capítulo 3
- 📘 *Pattern Recognition and Machine Learning* (Bishop) - Capítulo 3
- 📘 *Deep Learning* (Goodfellow et al.) - Capítulo 5

**Cursos:**
- 🎓 Andrew Ng - Machine Learning (Coursera)
- 🎓 Stanford CS229 - Machine Learning
- 🎓 Fast.ai - Practical Deep Learning

**Práctica:**
- 💻 Kaggle: House Prices Competition
- 💻 UCI ML Repository: Diversos datasets
- 💻 Implementa variantes (Ridge, LASSO) desde cero

---

### Reflexión Final

> *"Todos los modelos son incorrectos, pero algunos son útiles"* - George Box

Regresión lineal es "incorrecto" (el mundo rara vez es perfectamente lineal), pero es **tremendamente útil**:
- ✅ Rápido de entrenar y evaluar
- ✅ Fácil de interpretar y explicar a stakeholders
- ✅ Robusto cuando se aplica correctamente
- ✅ Base conceptual para técnicas avanzadas

No busques el modelo perfecto. Busca el **modelo más simple que resuelva tu problema**.

**¡Felicitaciones por completar este notebook! 🎉**

---
## 10. 📚 Referencias y Navegación

### Ruta 1: ML Clásico

1. **[01 - Regresión Lineal](01-regresion-lineal.ipynb)** ⭐ Estás aquí
2. [02 - Gradient Descent](02-gradient-descent.ipynb)
3. [03 - Regresión Logística](03-regresion-logistica.ipynb)
4. [04 - Regresión Softmax](04-regresion-softmax.ipynb)
5. [05 - Árboles de Decisión](05-arboles-decision.ipynb)
6. [06 - Random Forests](06-random-forests.ipynb)
7. [07 - Boosting](07-boosting.ipynb)
8. [08 - SVM](08-svm.ipynb)
9. [09 - K-Means](09-kmeans.ipynb)
10. [10 - PCA](10-pca.ipynb)

---

**[02 - Gradient Descent ➡️](02-gradient-descent.ipynb)**

**[🏠 Volver al índice principal](../../README.md)**

---

### Referencias Bibliográficas

1. Legendre, A. M. (1805). *Nouvelles méthodes pour la détermination des orbites des comètes*.
2. Gauss, C. F. (1809). *Theoria motus corporum coelestium*.
3. Hoerl, A. E., & Kennard, R. W. (1970). *Ridge regression: Biased estimation for nonorthogonal problems*.
4. Tibshirani, R. (1996). *Regression shrinkage and selection via the lasso*.
5. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The elements of statistical learning*.
6. Bishop, C. M. (2006). *Pattern recognition and machine learning*.
7. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep learning*.

---

**Última actualización:** 2024  
**Autor:** Equipo de Tutoriales AI  
**Licencia:** MIT

---

💡 **¿Encontraste un error o tienes sugerencias?**  
Abre un issue en nuestro repositorio de GitHub.